In [ ]:
# --- IMPORTS ---
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

from IPython.display import display
from google.colab import files
import io

# --- FILE UPLOAD ---
uploaded = files.upload()
df = pd.read_excel(io.BytesIO(uploaded[list(uploaded.keys())[0]]))

# --- MISSING VALUE HANDLING ---
print("Missing values in dataset:")
print(df.isnull().sum())

# Fill missing values
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
df[categorical_cols] = df[categorical_cols].fillna(df[categorical_cols].mode().iloc[0])

# --- ENCODE CATEGORICAL VARIABLES ---
categorical_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating',
                    'airconditioning', 'prefarea', 'furnishingstatus']

label_encoders = {col: LabelEncoder() for col in categorical_cols}
for col, encoder in label_encoders.items():
    df[col] = encoder.fit_transform(df[col])

# --- VISUALIZATION: PRICE vs AREA ---
plt.figure(figsize=(6, 4))
sns.scatterplot(data=df, x='area', y='price', hue='furnishingstatus', palette='Set2')
plt.xlabel('Area')
plt.ylabel('Price')
plt.title('Price vs Area Scatter Plot')
plt.grid(True)
plt.tight_layout()
plt.show()

# --- CORRELATION HEATMAP ---
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix of Housing Features')
plt.tight_layout()
plt.show()

# --- FEATURE SELECTION ---
X = df.drop('price', axis=1)
y = df['price']

# --- SPLIT DATA ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- PREPROCESS NUMERIC FEATURES ---
numerical_cols = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols)
], remainder='passthrough')

# --- DEFINE REGRESSION MODELS ---
models = {
    "Random Forest": RandomForestRegressor(random_state=42, n_estimators=100),
    "Linear Regression": LinearRegression(),
    "K-Nearest Neighbors": KNeighborsRegressor(n_neighbors=5),
    "Support Vector Regression": SVR(kernel='linear')
}

# --- EVALUATE MODELS ---
results = []
for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    results.append({
        "Model": name,
        "R2 Score": r2_score(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred))
    })

# --- RESULTS TABLE ---
results_df = pd.DataFrame(results)
display(results_df)

# --- GROUPED BAR PLOT FOR MULTIPLE METRICS ---
# Reshape for grouped barplot
melted_df = results_df.melt(id_vars="Model",
                            value_vars=["R2 Score", "MAE", "RMSE"],
                            var_name="Metric",
                            value_name="Value")

plt.figure(figsize=(10, 6))
sns.barplot(data=melted_df, x="Model", y="Value", hue="Metric", palette="Set2")
plt.title("Comparison of Regression Models by Metrics")
plt.ylabel("Score")
plt.xticks(rotation=45)
plt.legend(title="Metric")
plt.tight_layout()
plt.show()

: 